## 1. Upload the code
Run this and select `pdc-osfed-colab.zip`. It extracts the project and makes it ready.

In [1]:
from google.colab import files
up = files.upload()          # yahan "pdc-osfed-colab.zip" select karein
import zipfile, os
fn = list(up.keys())[0]
with zipfile.ZipFile(fn) as z:
    z.extractall('/content/pdc-osfed')
%cd /content/pdc-osfed
print(os.listdir())

Saving pdc-osfed-colab.zip to pdc-osfed-colab.zip
/content/pdc-osfed
['src', 'scripts', 'results', 'notes', '.gitignore', 'requirements.txt', 'logs', 'data', 'configs']


## 2. Install pyyaml
Colab already has torch/torchvision; we only need pyyaml.

In [2]:
!pip -q install pyyaml

## 3. Snapshot tuning — grid 1 (Step 8.10)
One client, one round. `match_loss` should be small and `cos_sim` clearly positive.

In [3]:
!python -m scripts.tune_snapshot --config configs/A_osfed_cnn_noniid_g1ec0_smoke_s1.yaml

device       : cuda
config       : configs/A_osfed_cnn_noniid_g1ec0_smoke_s1.yaml
model/grid   : cnn  grid=1x1
client id    : 0
inits        : ['first_client_image', 'random']
L-BFGS iters : [10, 20, 50]
100% 170M/170M [36:59<00:00, 76.8kB/s]
client samples: 500
local train  : loss=1.4103 time=1.7s  ||dW||=0.4039  dims=545098

init                 iters    match_loss   cos_sim  resid_ratio   opt_s
-----------------------------------------------------------------------
first_client_image      10      0.786994    0.4615       0.8871    0.42
first_client_image      20      0.713907    0.5355       0.8449    0.14
first_client_image      50      0.691767    0.5552       0.8317    0.41
random                  10      0.747050    0.5029       0.8643    0.06
random                  20      0.769806    0.4868       0.8774    0.12
random                  50      0.683682    0.5810       0.8269    0.39

saved: /content/pdc-osfed/results/E0_sanity/tune_snapshot_cnn_g1_client0.csv

How to read this

## 4. Snapshot tuning — grid 4 (CIFAR default)
Multi-grid gives better matching (cos_sim ~0.77 vs ~0.58 at grid 1).

In [4]:
# grid-4 config banao aur tuning chalao
import yaml
c = yaml.safe_load(open('configs/A_osfed_cnn_noniid_g1ec0_smoke_s1.yaml'))
c['grid_size'] = 4
yaml.safe_dump(c, open('configs/A_osfed_cnn_noniid_g4ec0_smoke_s1.yaml','w'), sort_keys=False)
!python -m scripts.tune_snapshot --config configs/A_osfed_cnn_noniid_g4ec0_smoke_s1.yaml

device       : cuda
config       : configs/A_osfed_cnn_noniid_g4ec0_smoke_s1.yaml
model/grid   : cnn  grid=4x4
client id    : 0
inits        : ['first_client_image', 'random']
L-BFGS iters : [10, 20, 50]
client samples: 500
local train  : loss=1.4103 time=0.6s  ||dW||=0.4039  dims=545098

init                 iters    match_loss   cos_sim  resid_ratio   opt_s
-----------------------------------------------------------------------
first_client_image      10      0.566486    0.6586       0.7527    0.31
first_client_image      20      0.503799    0.7046       0.7098    0.18
first_client_image      50      0.453123    0.7399       0.6731    0.53
random                  10      0.536258    0.6810       0.7323    0.08
random                  20      0.497085    0.7092       0.7050    0.16
random                  50      0.411729    0.7672       0.6417    0.52

saved: /content/pdc-osfed/results/E0_sanity/tune_snapshot_cnn_g4_client0.csv

How to read this (plan Step 8.10):
  * Pick the setting

## 5. OS-Fed smoke — grid 4, error compensation ON (ec1), 20 rounds
Pipeline check: test accuracy should rise over rounds.

In [5]:
import yaml
c = yaml.safe_load(open('configs/A_osfed_cnn_noniid_g4ec0_smoke_s1.yaml'))
c.update(purpose="OS-Fed g4 ec1 smoke: 10 clients, 20 rounds",
         n_clients=10, samples_per_client=None, rounds=20,
         grid_size=4, error_compensation=True,
         snapshot_init="random", lbfgs_max_iter=50)
yaml.safe_dump(c, open('configs/A_osfed_cnn_noniid_g4ec1_s1.yaml','w'), sort_keys=False)
!python -m src.run_single --config configs/A_osfed_cnn_noniid_g4ec1_s1.yaml

run_id : 20261124_A_osfed_cnn_noniid_g4ec1_s1__net0
device : cuda
method : osfed
purpose: OS-Fed g4 ec1 smoke: 10 clients, 20 rounds
client sizes: [3701, 5607, 1996, 4294, 5058, 5869, 2650, 3518, 6244, 11063]
[round   1/20] lr=0.01000 train_loss=1.3236 test_acc=0.1735 test_loss=2.2341 up=0.129MB down=21.804MB round=16.0s total=0.3min
[round   2/20] lr=0.00994 train_loss=1.2416 test_acc=0.2558 test_loss=2.1824 up=0.129MB down=21.804MB round=15.1s total=0.5min
[round   3/20] lr=0.00976 train_loss=1.1321 test_acc=0.2637 test_loss=2.2654 up=0.129MB down=21.804MB round=15.0s total=0.8min
[round   4/20] lr=0.00946 train_loss=1.0920 test_acc=0.3356 test_loss=1.9458 up=0.129MB down=21.804MB round=15.3s total=1.0min
[round   5/20] lr=0.00905 train_loss=1.0594 test_acc=0.3474 test_loss=1.8875 up=0.129MB down=21.804MB round=15.2s total=1.3min
[round   6/20] lr=0.00854 train_loss=1.0275 test_acc=0.3272 test_loss=2.0702 up=0.129MB down=21.804MB round=15.2s total=1.5min
[round   7/20] lr=0.00794 tra

## 6. OS-Fed smoke — grid 4, error compensation OFF (ec0), 20 rounds
To compare ec1 vs ec0.

In [6]:
import yaml
c = yaml.safe_load(open('configs/A_osfed_cnn_noniid_g4ec1_s1.yaml'))
c.update(purpose="OS-Fed g4 ec0 (no error comp): 10 clients, 20 rounds",
         error_compensation=False)
yaml.safe_dump(c, open('configs/A_osfed_cnn_noniid_g4ec0_s1.yaml','w'), sort_keys=False)
!python -m src.run_single --config configs/A_osfed_cnn_noniid_g4ec0_s1.yaml

run_id : 20261124_A_osfed_cnn_noniid_g4ec0_s1__net0
device : cuda
method : osfed
purpose: OS-Fed g4 ec0 (no error comp): 10 clients, 20 rounds
client sizes: [3701, 5607, 1996, 4294, 5058, 5869, 2650, 3518, 6244, 11063]
[round   1/20] lr=0.01000 train_loss=1.3236 test_acc=0.1801 test_loss=2.2357 up=0.129MB down=21.804MB round=16.5s total=0.3min
[round   2/20] lr=0.00994 train_loss=1.2424 test_acc=0.2602 test_loss=2.0364 up=0.129MB down=21.804MB round=15.3s total=0.5min
[round   3/20] lr=0.00976 train_loss=1.1524 test_acc=0.3003 test_loss=1.9249 up=0.129MB down=21.804MB round=15.4s total=0.8min
[round   4/20] lr=0.00946 train_loss=1.1124 test_acc=0.3053 test_loss=1.8830 up=0.129MB down=21.804MB round=15.8s total=1.0min
[round   5/20] lr=0.00905 train_loss=1.0891 test_acc=0.3150 test_loss=1.9303 up=0.129MB down=21.804MB round=15.6s total=1.3min
[round   6/20] lr=0.00854 train_loss=1.0755 test_acc=0.3212 test_loss=1.8822 up=0.129MB down=21.804MB round=15.3s total=1.6min
[round   7/20] lr=0

## 7. OS-Fed MAIN — ec1 (error compensation ON), 100 rounds
**NOTE: This run DIVERGED (NaN around round ~58).** This is a documented finding (see `notes/decisions.md`). That is why our stable result uses ec0 (cell 9). Do not delete this output — it is the evidence.

In [7]:
import yaml, shutil
c = yaml.safe_load(open('configs/A_osfed_cnn_noniid_g4ec1_s1.yaml'))
c.update(purpose="OS-Fed MAIN E1: g4 ec1, 10 clients, 100 rounds", rounds=100)
yaml.safe_dump(c, open('configs/A_osfed_cnn_noniid_g4ec1_s1.yaml','w'), sort_keys=False)
# pehle wala 20-round (smoke) hata do — uske numbers notes mein save hain
for d in ['logs/20261124_A_osfed_cnn_noniid_g4ec1_s1__net0',
          'checkpoints/20261124_A_osfed_cnn_noniid_g4ec1_s1__net0']:
    shutil.rmtree(d, ignore_errors=True)
!python -m src.run_single --config configs/A_osfed_cnn_noniid_g4ec1_s1.yaml

run_id : 20261124_A_osfed_cnn_noniid_g4ec1_s1__net0
device : cuda
method : osfed
purpose: OS-Fed MAIN E1: g4 ec1, 10 clients, 100 rounds
client sizes: [3701, 5607, 1996, 4294, 5058, 5869, 2650, 3518, 6244, 11063]
[round   1/100] lr=0.01000 train_loss=1.3236 test_acc=0.1843 test_loss=2.2346 up=0.129MB down=21.804MB round=15.8s total=0.3min
[round   2/100] lr=0.01000 train_loss=1.2415 test_acc=0.2472 test_loss=2.2357 up=0.129MB down=21.804MB round=15.3s total=0.5min
[round   3/100] lr=0.00999 train_loss=1.1323 test_acc=0.2526 test_loss=2.2852 up=0.129MB down=21.804MB round=15.1s total=0.8min
[round   4/100] lr=0.00998 train_loss=1.0900 test_acc=0.3257 test_loss=1.9400 up=0.129MB down=21.804MB round=15.2s total=1.0min
[round   5/100] lr=0.00996 train_loss=1.0572 test_acc=0.3034 test_loss=2.0505 up=0.129MB down=21.804MB round=15.2s total=1.3min
[round   6/100] lr=0.00994 train_loss=1.0249 test_acc=0.2697 test_loss=2.0467 up=0.129MB down=21.804MB round=15.2s total=1.5min
[round   7/100] lr=

## 8. Download results (zip)
logs/results/configs/notes in one zip, to copy into the local project.

In [8]:
import zipfile, glob
from google.colab import files
with zipfile.ZipFile('/content/osfed_outputs.zip','w',zipfile.ZIP_DEFLATED) as z:
    for pat in ['logs/**/*','results/**/*','configs/*osfed*',
                'notes/decisions.md','notes/run_log.md','notes/env_colab.txt']:
        for f in glob.glob(pat, recursive=True):
            z.write(f)
files.download('/content/osfed_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. OS-Fed MAIN (STABLE) — ec0 (error compensation OFF), 100 rounds
**This is our real OS-Fed result** (final test_acc ~0.4350). No NaN, stable.

In [9]:
import yaml, shutil
c = yaml.safe_load(open('configs/A_osfed_cnn_noniid_g4ec0_s1.yaml'))
c.update(purpose="OS-Fed MAIN E1 (stable, no error comp): g4 ec0, 100 rounds", rounds=100)
yaml.safe_dump(c, open('configs/A_osfed_cnn_noniid_g4ec0_s1.yaml','w'), sort_keys=False)
for d in ['logs/20261124_A_osfed_cnn_noniid_g4ec0_s1__net0',
          'checkpoints/20261124_A_osfed_cnn_noniid_g4ec0_s1__net0']:
    shutil.rmtree(d, ignore_errors=True)
!python -m src.run_single --config configs/A_osfed_cnn_noniid_g4ec0_s1.yaml

run_id : 20261124_A_osfed_cnn_noniid_g4ec0_s1__net0
device : cuda
method : osfed
purpose: OS-Fed MAIN E1 (stable, no error comp): g4 ec0, 100 rounds
client sizes: [3701, 5607, 1996, 4294, 5058, 5869, 2650, 3518, 6244, 11063]
[round   1/100] lr=0.01000 train_loss=1.3236 test_acc=0.1641 test_loss=2.2371 up=0.129MB down=21.804MB round=16.6s total=0.3min
[round   2/100] lr=0.01000 train_loss=1.2412 test_acc=0.2686 test_loss=2.0321 up=0.129MB down=21.804MB round=15.2s total=0.5min
[round   3/100] lr=0.00999 train_loss=1.1484 test_acc=0.3028 test_loss=1.9177 up=0.129MB down=21.804MB round=15.3s total=0.8min
[round   4/100] lr=0.00998 train_loss=1.1074 test_acc=0.3131 test_loss=1.8766 up=0.129MB down=21.804MB round=15.5s total=1.0min
[round   5/100] lr=0.00996 train_loss=1.0842 test_acc=0.3236 test_loss=1.8996 up=0.129MB down=21.804MB round=15.8s total=1.3min
[round   6/100] lr=0.00994 train_loss=1.0688 test_acc=0.3273 test_loss=1.8687 up=0.129MB down=21.804MB round=15.8s total=1.6min
[round 

## 10. Download results (zip) — again
To download the new logs after the stable (ec0) run.

In [10]:
import zipfile, glob
from google.colab import files
with zipfile.ZipFile('/content/osfed_outputs.zip','w',zipfile.ZIP_DEFLATED) as z:
    for pat in ['logs/**/*','results/**/*','configs/*osfed*',
                'notes/decisions.md','notes/run_log.md','notes/env_colab.txt']:
        for f in glob.glob(pat, recursive=True):
            z.write(f)
files.download('/content/osfed_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>